In [1]:
 from AIP_interaction_map.asc import calculate_association_energy as ac
from AIP_interaction_map.asc import calculate_binding_energy as bc
from AIP_interaction_map.asc import calculate_solvation_energy as sc
from AIP_interaction_map.PDB_preprocess import PDB_preprocess 
from AIP_interaction_map.getScoring import Scoring
from AIP_interaction_map.scoring_traj_hg import ScoringTraj
from AIP_interaction_map.create_jmol_vis import create_jmol_vis
from AIP_interaction_map.create_jmol_vis import create_ip_vis
from AIP_interaction_map.get_desolvation import get_desolvated_AIPs, get_free_energy_of_solvation
import mdtraj as md
import pandas as pd
import numpy as np

In [8]:
comps = ["comp41", "comp42"]
path = "/home/kate/workspace/Orig_code_and_datasets/S30L/"

PDB_file1 = f"{path}/{comps[0]}/{comps[0]}.pdb"
PDB_file2 = f"{path}/{comps[1]}/{comps[1]}.pdb"
PDB_preprocess(PDB_file1, PDB_file2)

In [ ]:
self = Scoring(PDB_file1, PDB_file2, protein_host=False, max_aip_dist=0.15,  solvent="chloroform")
#create_ip_vis(self, f"{comps[0]}_{comps[1]}")

In [8]:
df = pd.read_csv('/home/kate/workspace/Orig_code_and_datasets/comp41_42/comp41_42.csv')
df

,Unnamed: 0,comp41,comp41_type,comp41_value,comp42,comp42_type,comp42_value,Frac,ddG (kJ/mol)
0,0,14.0,H.soft,0.5,63.0,N.pl3.am,-2.6,0.5,-1.0
1,1,14.0,H.soft,0.5,112.0,C.2,-3.3,0.5,-0.3
2,2,15.0,H.soft,0.6,47.0,N.pl3.am,-2.6,0.5,-1.0
3,3,15.0,H.soft,0.6,60.0,N.pl3.am,-2.2,0.5,-1.3
4,4,17.0,H.soft,0.5,45.0,N.pl3.am,-2.2,0.5,-1.3
5,5,22.0,H.soft,0.5,48.0,N.pl3.am,-2.2,0.5,-1.3
6,6,24.0,H.soft,0.4,93.0,C.2,-3.5,0.5,-0.1
7,7,24.0,H.soft,0.4,97.0,C.2,-3.5,0.5,-0.1
8,8,12.0,H.soft,0.6,65.0,N.pl3.am,-2.2,0.5,-1.3
9,9,12.0,H.soft,0.6,109.0,C.2,-3.6,0.5,-0.1


In [9]:
frac = df.Frac
dG1 = sum(df["ddG (kJ/mol)"])
dG2 = sum([bc(j[3], j[6], "chloroform")*frac[i] for i,j in df.iterrows()])
dG3 = sum([j[3]*j[7]*j[6] for _,j in df.iterrows()])
dG4 = -sum([sc(j[3], j[6], "chloroform")*frac[i] for i,j in df.iterrows()])
dG5 = sum([sc(j[3], j[6], "n-hexadecane")*frac[i] for i,j in df.iterrows()])
dG6 = sum([sc(j[3], j[6], "noble")*frac[i] for i,j in df.iterrows()])

In [10]:
for i,j in df.iterrows():
    solv_host = sc(j[6], None, "water")
    solv_g = sc(j[3], None, "water")
    print(round(df["ddG (kJ/mol)"][i]+solv_host+solv_g,2))

-2.44
-3.24
-2.55
-2.12
-2.0
-2.0
-3.41
-3.41
-2.12
-3.86
-2.32
-2.51
-9.5
-2.26
-3.51
-3.86
-3.37
-3.86


In [27]:
#with repulsion
print("SSIMPLE", round(dG1,2))
print("binding energy", round(dG2,2))
print("a.b product", round(dG3,2))
print("chloroform desolv", round(dG4,2))
print("hex solv", round(dG5,2))
print("noble solv", round(dG6,2))
print("A.B+PhTr", round(dG3+dG4+dG6,2))

SSIMPLE 1.4
binding energy -18.86
a.b product -4.22
chloroform desolv 30.04
hex solv -18.82
noble solv -16.36
A.B+PhTr 9.46
